In [1]:
import itertools
import yaml
import torch
from pathlib import Path
from neuralhydrology.nh_run import start_run

import pickle
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# ------------ Paths ------------
runs_dir = Path("./runs")
ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)
ensemble_timeseries_dir=Path("./ensemble_timeseries")
ensemble_timeseries_dir.mkdir(exist_ok=True)
hydrographs_dir = Path("./hydrographs")

In [3]:
precip_products = [
    "prcp_mm_day",
    "prcp_chirps_mm_day",
    "prcp_mswep_mm_day",
    "prcp_gauge_mm_day",
]

precip_combo = list(itertools.combinations(precip_products, 4))[0]

seeds = [111, 222, 333, 444, 555, 666, 777, 888]

In [4]:
base_config_path = Path("training_new_wy.yml")

with open(base_config_path, "r") as f:
    base_config = yaml.safe_load(f)

In [5]:
for seed in seeds:

    config = base_config.copy()

    # Update seed
    config["seed"] = seed

    # Update dynamic inputs:
    # Keep temperature and radiation, replace precip
    config["dynamic_inputs"] = [
        "srad_W_m2",
        "tmax_C",
        "tmin_C",
        *precip_combo
    ]

    # Create unique experiment name
    precip_name = "_".join(precip_combo)
    config["experiment_name"] = (
        f"new_wy_precip_{precip_name}_seed_{seed}"
    )

    # Save temporary config file
    temp_config_path = Path(f"temp_{precip_name}_seed_{seed}.yml")
    with open(temp_config_path, "w") as f:
        yaml.dump(config, f)

    print(f"Running: {config['experiment_name']}")

    # Run training
    if torch.cuda.is_available() or torch.backends.mps.is_available():
        start_run(config_file=temp_config_path)
    else:
        start_run(config_file=temp_config_path, gpu=-1)

Running: new_wy_precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111
2026-02-26 22:03:18,405: Logging to /home/azureuser/sky_workdir/final_runs/runs/new_wy_precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2602_220318/output.log initialized.
2026-02-26 22:03:18,406: ### Folder structure created at /home/azureuser/sky_workdir/final_runs/runs/new_wy_precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2602_220318
2026-02-26 22:03:18,406: ### Run configurations for new_wy_precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111
2026-02-26 22:03:18,407: batch_size: 256
2026-02-26 22:03:18,407: clip_gradient_norm: 1
2026-02-26 22:03:18,408: data_dir: data
2026-02-26 22:03:18,408: dataset: generic
2026-02-26 22:03:18,409: device: cuda:0
2026-02-26 22:03:18,409: dynamic_inputs: ['srad_W_m2', 'tmax_C', 'tmin_C', 'prcp_mm_day', 'prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'prcp_g

# Ensemble

In [8]:
def nse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    denominator = ((obs - obs.mean())**2).sum()
    numerator = ((sim - obs)**2).sum()

    return float(1 - numerator / denominator)


def mse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate mean squared error."""
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    return float(((sim - obs)**2).mean())


def rmse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate root mean squared error."""
    return np.sqrt(mse(obs, sim))


def beta_kge(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate the beta KGE term (ratio of means)."""
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    return float(sim.mean() / obs.mean())

In [9]:
precip_name = "_".join(precip_combo)
print(f"Processing: new_wy_{precip_name}")

runs_list = list(runs_dir.glob(f"new_wy_precip_{precip_name}_seed*"))

# if not runs_list:
#     print(f"  No runs found, skipping...")
#     continue

all_seeds_dfs = []

for run in runs_list:
    seed = run.name.split("_seed_")[1].split("_")[0]
    pickle_path = run / "validation" / "model_epoch030" / "validation_results.p"
    
    if not pickle_path.exists():
        continue
        
    with open(pickle_path, "rb") as f:
        data = pickle.load(f)
    
    basin_dfs = []
    
    for basin in data.keys():
        ds = data[basin]['1D']['xr']
        df = ds.to_dataframe().reset_index()
        df = df[df["time_step"] == 0]
        df["basin"] = basin
        df = df.rename(columns={"QObs_mm_d_sim": f"Q_sim_seed_{seed}"})
        df = df[["date", "basin", "QObs_mm_d_obs", f"Q_sim_seed_{seed}"]]
        basin_dfs.append(df)
    
    seed_df = pd.concat(basin_dfs, ignore_index=True)
    all_seeds_dfs.append(seed_df)

# if not all_seeds_dfs:
#     print(f"  No valid data, skipping...")
#     continue

# Start with first seed
results_df = all_seeds_dfs[0]

# Merge remaining seeds
for df in all_seeds_dfs[1:]:
    sim_col = [col for col in df.columns if "Q_sim_seed" in col][0]
    results_df = results_df.merge(
        df[["date", "basin", sim_col]],
        on=["date", "basin"],
        how="left"
    )

# Calculate mean
sim_cols = [col for col in results_df.columns if "Q_sim_seed" in col]
results_df["Q_sim_mean"] = results_df[sim_cols].mean(axis=1)

results_df.to_csv(ensemble_timeseries_dir / f"new_wy_{precip_name}_timeseries.csv", index=False)

# Compute metrics for each basin
metrics_data = []

for basin, df_basin in results_df.groupby("basin"):
    obs = xr.DataArray(
        df_basin["QObs_mm_d_obs"].values,
        dims=["time"],
        coords={"time": df_basin["date"].values}
    )
    sim = xr.DataArray(
        df_basin["Q_sim_mean"].values,
        dims=["time"],
        coords={"time": df_basin["date"].values}
    )
    
    metrics_data.append({
        "basin": basin,
        "NSE": nse(obs, sim),
        "RMSE": rmse(obs, sim),
        "MSE": mse(obs, sim),
        "Beta-KGE": beta_kge(obs, sim)
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv(ensemble_metrics_dir / f"new_wy_{precip_name}.csv", index=False)
print(f"  Saved: new_wy_{precip_name}.csv")

Processing: new_wy_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day
  Saved: new_wy_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day.csv
